# Home work
1. Write a function that:
    - Takes a block number as input
    - Returns the timestamp of when it was mined (human-readable)
    - Hint: We did this in the ERC20 transfer event analysis section
2. Calculate average gas price of last 10 blocks
3. Create a simple whale detector function
    - The function should:
        - Check if a wallet has more than 100 ETH
        - Check if a wallet has more than 1M USDC
        - If either is True, the wallet should be tagged a "Whale" (can simply print "Whale")
        - Optionally, can add more categories (goldfish, dolphin, small whale, large whale, etc.)
    - Test the function with your address or addresses you find on [etherscan.io](https://etherscan.io/token/0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48#balances)
    - Bonus:
        - Create a function that takes in several addresses, checks its category, and appends the address and tag to a dictionary (Address is key, tag is the value)

In [5]:
from web3 import Web3
from datetime import datetime
from dotenv import load_dotenv
import os

In [6]:
# 1. Load environment variables from the .env file
load_dotenv()
GATEWAY = os.getenv("GATEWAY_URL")

print(f'GATEWAY imported as {GATEWAY}') # debugging to find out if our variables have been loaded

GATEWAY imported as https://mainnet.infura.io/v3/08713ab1843242a9a95fc0d4cecae905


In [7]:
# Next is to connect to a blockchain(Ethereum node) using my provided URL

GATEWAY = "https://mainnet.infura.io/v3/08713ab1843242a9a95fc0d4cecae905"
w3 = Web3(Web3.HTTPProvider(GATEWAY))
print(f'Is the connection successful?: {w3.is_connected()}') # Should return True,print statement for debugging purpose too

Is the connection successful?: True


In [ ]:
# Function that takes in block number and returns timestamp when block was mined in human readable format(string)

def get_block_timestamp(block_number):
    # Fetch block details by block number
    block = w3.eth.get_block(block_number)

    # The timestamp in the block is in Unix time (seconds since epoch)
    timestamp = block.timestamp

    # Convert Unix timestamp to human-readable datetime string
    readable_time = datetime.fromtimestamp(timestamp).strftime('%Y-%m-%d %H:%M:%S')

    return readable_time



In [16]:
# Test the above function, we are going to use the block number;23268375
from datetime import datetime

block_num = 23268375
print(f"Block {block_num} was mined at: {get_block_timestamp(block_num)}")


Block 23268375 was mined at: 2025-09-01 05:46:23


In [19]:
# Retrieving the last 10 blocks
R=10
latest_block = w3.eth.block_number
#  base_fees = []

for i in range(latest_block, latest_block - R, -1):
    print(f'fetching block {i}') 
    block = w3.eth.get_block(i)

fetching block 23270621
fetching block 23270620
fetching block 23270619
fetching block 23270618
fetching block 23270617
fetching block 23270616
fetching block 23270615
fetching block 23270614
fetching block 23270613
fetching block 23270612


In [32]:
# Create a function to help us calculate the average gas price of the last 10 blocks in GWEI

def average_gas_price_last_n_blocks(R=10):
    latest_block = w3.eth.block_number

    gas_prices = []

# at this point we loop backwards over the last R blocks
    for block_num in range(latest_block, latest_block - R, -1):
        print(f'fetching block {block_num}') 
        block = w3.eth.get_block(block_num, full_transactions=True)
# for each transaction, we collect its gas price in GWEI
        for tx in block.transactions:
            gas_price_gwei = tx.gasPrice / 10**9  # convert Wei to Gwei
            gas_prices.append(gas_price_gwei)

    if not gas_prices:
        return 0

    avg_gas_price = sum(gas_prices) / len(gas_prices)
    return avg_gas_price        
    

In [33]:
# Testing to find out whether our function works
R=10
avg_gas = average_gas_price_last_n_blocks(R)
print(f"Average gas price of last {R} blocks: {avg_gas:.2f} Gwei")

fetching block 23270722
fetching block 23270721
fetching block 23270720
fetching block 23270719
fetching block 23270718
fetching block 23270717
fetching block 23270716
fetching block 23270715
fetching block 23270714
fetching block 23270713
Average gas price of last 10 blocks: 2.98 Gwei


In [ ]:
# Creating a simple whale detector function

# USDC contract address on Ethereum Mainnet
USDC_ADDRESS = Web3.to_checksum_address("0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48")

# Minimal ABI to get the balance of an ERC-20 token
ERC20_ABI = [
    {
        "constant": True,
        "inputs": [{"name": "_owner", "type": "address"}],
        "name": "balanceOf",
        "outputs": [{"name": "balance", "type": "uint256"}],
        "type": "function",
    },
    {
        "constant": True,
        "inputs": [],
        "name": "decimals",
        "outputs": [{"name": "", "type": "uint8"}],
        "type": "function",
    },
]

# Initializing USDC contract

usdc_contract = w3.eth.contract(address=USDC_ADDRESS, abi=ERC20_ABI)

def check_wallet_category(wallet_address):

    wallet_address = Web3.to_checksum_address(wallet_address)
    
    # Getting ETH balance (in Wei), converting to ETH
    eth_balance = w3.eth.get_balance(wallet_address) / 10**18
    
    # Getting USDC balance (in smallest unit), converting using decimals
    usdc_balance_raw = usdc_contract.functions.balanceOf(wallet_address).call()
    decimals = usdc_contract.functions.decimals().call()
    usdc_balance = usdc_balance_raw / (10 ** decimals)
    
    print(f"ETH Balance: {eth_balance:.4f} ETH")
    print(f"USDC Balance: {usdc_balance:.2f} USDC")
    
    # Determining category based on ETH and USDC holdings
    if eth_balance > 100 or usdc_balance > 1_000_000:
        print("Tag: Whale")
    elif 50 < eth_balance <= 100:
        print("Tag: Large Whale")
    elif 10 < eth_balance <= 50:
        print("Tag: Small Whale")
    elif 1 < eth_balance <= 10:
        print("Tag: Dolphin")
    elif eth_balance <= 1:
        print("Tag: Goldfish")
    else:
        print("Tag: Unknown category")


In [ ]:
# Testing the above whale function
wallet = "0xd40E9C8cE75615C91e6D3c56f9A16C65A6BD3b35"  # example wallet
check_wallet_category(wallet)

ETH Balance: 8.1696 ETH
USDC Balance: 6433.81 USDC
Tag: Dolphin


In [38]:
# Testing the above whale function
wallet = "0xab5801a7d398351b8be11c439e05c5b3259aec9b"  # example wallet
check_wallet_category(wallet)

ETH Balance: 0.0781 ETH
USDC Balance: 2568.56 USDC
Tag: Goldfish


In [40]:
# Testing the above whale function
wallet = "0x6cc8dCbCA746a6E4Fdefb98E1d0DF903b107fd21"  # example wallet
check_wallet_category(wallet)

ETH Balance: 61.7051 ETH
USDC Balance: 129544.02 USDC
Tag: Large Whale


In [42]:
# Testing the above whale function
wallet = "0x4E5B2e1dc63F6b91cb6Cd759936495434C7e972F"  # example wallet
check_wallet_category(wallet)

ETH Balance: 1272.2738 ETH
USDC Balance: 1054723.74 USDC
Tag: Whale


In [45]:
# Testing the above whale function
wallet = "0xdadB0d80178819F2319190D340ce9A924f783711"  # example wallet
check_wallet_category(wallet)

ETH Balance: 33.8793 ETH
USDC Balance: 0.00 USDC
Tag: Small Whale


In [53]:
# Function that takes in several addresses and checks whether they are a whale, large whale, small whale, dolpin or gold fish

USDC_ADDRESS = Web3.to_checksum_address("0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48")
ERC20_ABI = [
    {
        "constant": True,
        "inputs": [{"name": "_owner", "type": "address"}],
        "name": "balanceOf",
        "outputs": [{"name": "balance", "type": "uint256"}],
        "type": "function",
    },
    {
        "constant": True,
        "inputs": [],
        "name": "decimals",
        "outputs": [{"name": "", "type": "uint8"}],
        "type": "function",
    },
]
usdc_contract = w3.eth.contract(address=USDC_ADDRESS, abi=ERC20_ABI)

def get_wallet_tag(wallet_address):
    wallet_address = Web3.to_checksum_address(wallet_address)
    eth_balance = w3.eth.get_balance(wallet_address) / 10**18
    usdc_balance_raw = usdc_contract.functions.balanceOf(wallet_address).call()
    decimals = usdc_contract.functions.decimals().call()
    usdc_balance = usdc_balance_raw / (10 ** decimals)

    # Determining category based on ETH and USDC balances
    if eth_balance > 100 or usdc_balance > 1_000_000:
        return "Whale"
    elif 50 < eth_balance <= 100:
        return "Large Whale"
    elif 10 < eth_balance <= 50:
        return "Small Whale"
    elif 1 < eth_balance <= 10:
        return "Dolphin"
    elif eth_balance <= 1:
        return "Goldfish"
    else:
        return "Unknown"

def classify_wallets(wallet_addresses):
    result = {}
    for addr in wallet_addresses:
        tag = get_wallet_tag(addr)
        result[addr] = tag
    return result

In [55]:
# Testing function

wallets = [
    "0x4E5B2e1dc63F6b91cb6Cd759936495434C7e972F", 
    "0x6cc8dCbCA746a6E4Fdefb98E1d0DF903b107fd21",
    "0xdadB0d80178819F2319190D340ce9A924f783711",
    "0xd40E9C8cE75615C91e6D3c56f9A16C65A6BD3b35",
    "0xab5801a7d398351b8be11c439e05c5b3259aec9b"   
    # Add more addresses to test
]

tags = classify_wallets(wallets)
print(tags)

{'0x4E5B2e1dc63F6b91cb6Cd759936495434C7e972F': 'Whale', '0x6cc8dCbCA746a6E4Fdefb98E1d0DF903b107fd21': 'Large Whale', '0xdadB0d80178819F2319190D340ce9A924f783711': 'Small Whale', '0xd40E9C8cE75615C91e6D3c56f9A16C65A6BD3b35': 'Dolphin', '0xab5801a7d398351b8be11c439e05c5b3259aec9b': 'Goldfish'}


https://docs.google.com/document/d/1hlIIocYFczy_ZtTTIgz5wJuzDzWZE98dGGDcdnAA4Hg/edit?usp=sharing